### VPC Flow Log Hunting: part of the DUNE project
Ingests VPC flow logs, labels known cloud provider IPs, tor IPs, and anything known you want to define in known_ips.csv. Flows are then aggregated into records with summed time spans, byte and packet counts. Flows are tagged north-south or east-west (tag-direction) and ingress or egress (flow-direction).
   

In [52]:
# --- cell 00 ---
import pandas as pd
import html
import ipaddress
from ipaddress import ip_address
import numpy as np
import os
import json
import re
import warnings
from functools import reduce
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

# plotly 6 FigureWidget: after a redraw the front end can report a trace delta with
# no uid, and plotly's handler dies with KeyError: 'uid'. Fill the uid in by trace
# position before plotly reads it. Guarded so re-running this cell doesn't double-wrap.

from plotly.basewidget import BaseFigureWidget
if not getattr(BaseFigureWidget, "_uid_patch", False):
    _orig_trace_deltas = BaseFigureWidget._handler_js2py_traceDeltas.func
    def _trace_deltas_with_uid(self, change):
        for i, d in enumerate((change["new"] or {}).get("trace_deltas") or []):
            if "uid" not in d and i < len(self.data):
                d["uid"] = self.data[i].uid
        return _orig_trace_deltas(self, change)
    BaseFigureWidget._handler_js2py_traceDeltas = _trace_deltas_with_uid
    BaseFigureWidget._uid_patch = True

In [59]:
# --- cell 01 ---
parquet_file = "vflows-enriched-637602092272-2026.parquet"
vflowsAgg = pd.read_parquet(parquet_file)   # Cymru-enriched sessions (flows-nbad-cymru cell 11)
vflowsAgg.shape

(574605, 47)

In [54]:
# --- cell 16a ---
# Column order for the session table in cell 17. Reorder lines to move a
# column; comment one out to hide it. Derived tag-* fields first, then the flow.
# "hours" and "mb" are computed by the table cells; every other name must be a
# vflowsAgg column. A name vflowsAgg doesn't have (e.g. flow-direction on v2/v3 logs,
# tag-label, which beaconing.ipynb sets, or the peer_* columns, which only the
# Cymru-enriched sessions from flows-nbad-cymru have) is skipped quietly by the table cells.

TABLE_COLS = [
    "tag-direction",     # north-south / east-west
    "tag-org",           # cloud provider / known_ips.csv name
    "tag-cloud",         # which reference list matched: aws, azure, akamai, ...
    "tag-service",       # port / protocol name, or "anomalous"
    "tag-label",         # behaviour tags, e.g. beaconing (beaconing.ipynb)
    "peer_label",        # external end's owner -- Cymru-enriched sessions only
    "peer_country",      # external end's country -- Cymru-enriched sessions only
    "start_time",
    "end_time",
    "hours",             # computed: end - start
    "srcaddr",
    "srcport",
    "dstaddr",
    "dstport",
    "protocol",
    "action",
    "flow-direction",    # v5 logs only
    "mb",                # computed: bytes / 1e6
    "packets",
    "records",
]

_TABLE_COMPUTED = ("hours", "mb")
_dupCols = sorted({c for c in TABLE_COLS if TABLE_COLS.count(c) > 1})
if _dupCols:
    print(f"!! listed more than once: {_dupCols}")
_later   = {"tag-label", "flow-direction", "peer_label", "peer_country"}   # may legitimately be absent here
_unknown = [c for c in TABLE_COLS if c not in vflowsAgg.columns and c not in _TABLE_COMPUTED and c not in _later]
if _unknown:
    print(f"!! not vflowsAgg columns (typo?), will be skipped: {_unknown}")
print(f"{len(TABLE_COLS)} table columns: {', '.join(TABLE_COLS)}")

# Table stylesheet and widget label width shared by the display cells (17, 21, 22), set
# here so those cells can run in any order.
# explicit cell padding: JupyterLab only pads its own DataFrame output, not HTML in widgets
_TBL_CSS = ("<style>.flowTbl{font:12px ui-monospace,monospace;border-collapse:collapse}"
            ".flowTbl th,.flowTbl td{padding:3px 12px;text-align:right;white-space:nowrap}"
            ".flowTbl thead th{border-bottom:1px solid #888}"
            ".flowTbl tbody tr:nth-child(even){background:rgba(128,128,128,.08)}</style>")
_wsty = {"description_width": "70px"}

18 table columns: tag-direction, tag-org, tag-cloud, tag-service, tag-label, start_time, end_time, hours, srcaddr, srcport, dstaddr, dstport, protocol, action, flow-direction, mb, packets, records


In [55]:
# --- cell 22 ---
# Sessions and bytes by tag-cloud (vflowsAgg) -- cell 21's chart, grouped by which
# reference list the public end matched: aws, azure, google, akamai, cloudflare, fastly,
# github, oracle, zscaler, tor. Two panels on one shared axis -- never one chart with two scales.
# Sorted by sessions. Colors follow the cloud, not its rank: the 5 busiest (over all of
# vflowsAgg, fixed when this cell runs) get their own color and keep it through every
# filter; others are light gray; mid gray = no cloud (the public end isn't in any list --
# known_ips.csv names alone don't count). Every bar is also labeled, so color is never
# the only cue. Own names throughout (_cld*, CLD_*), so it doesn't touch cell 21.
# North-south sessions only (tag-direction): east-west sessions have no public end, so
# they never carry a tag-cloud and would only pad "(no cloud)". Address-less rows
# (NODATA / SKIPDATA) are left out the same way.
# Checkboxes pick which tag-cloud values to include -- one per cloud in the data plus
# "(no cloud)" (tag-cloud empty), in alphabetical order, all ticked to start; "all" / "none" set every box.
# Filters redraw the chart (a plain figure, not a FigureWidget -- see _cldFigure).
# Opens on: action ACCEPT, every cloud ticked, x scale linear, top 10.
# The chart shows the top N clouds by sessions (CLD_TOP_N, "show" dropdown); the table
# and `cloudSummary` always hold every cloud.

if "tag-cloud" not in vflowsAgg.columns:
    raise KeyError("vflowsAgg has no 'tag-cloud' column -- re-run cells 12-14 (or everything from cell 08) first")
from plotly.subplots import make_subplots

# validated (dataviz validator, all pairs): blue, yellow, magenta, violet, aqua.
# Orange, green and red are left out -- the full 8 can't all be told apart as a set.
_CLD_PALETTE = ["#2a78d6", "#eda100", "#e87ba4", "#4a3aa7", "#1baf7a"]
_CLD_OTHER   = "#c3c2b7"                                                  # remaining clouds
_CLD_COLORS  = {"(no cloud)": "#898781"}                                    # muted gray
_cldRank = (vflowsAgg["tag-cloud"].dropna().value_counts().index[:len(_CLD_PALETTE)])
_CLD_COLORS.update(dict(zip(_cldRank, _CLD_PALETTE)))
_INK, _INK2, _GRID, _SURF = "#0b0b0b", "#52514e", "#e8e7e2", "#fcfcfb"
CLD_TOP_N = 10


def _fmtBytes(b):
    for u in ("B", "kB", "MB", "GB", "TB"):
        if abs(b) < 1000 or u == "TB":
            return f"{b:,.0f} {u}" if u == "B" else f"{b:,.1f} {u}"
        b /= 1000


def _cldFigure(names, colors, sessions, nbytes, cdata, log, title):
    """A plain (one-way) plotly figure. Drawn fresh on every change -- no FigureWidget,
    whose two-way sync with the VS Code front end can hang the notebook."""
    fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.10,
                        subplot_titles=("Sessions", "Bytes"))
    for col, vals, txt in ((1, sessions, [f"{v:,}" for v in sessions]),
                           (2, nbytes,   [_fmtBytes(v) for v in nbytes])):
        fig.add_bar(row=1, col=col, orientation="h", y=names, x=vals, text=txt, customdata=cdata,
                    marker=dict(color=colors, cornerradius=4), textposition="outside", cliponaxis=False,
                    textfont_color=_INK2, showlegend=False,
                    hovertemplate="<b>%{y}</b><br>%{customdata[0]:,} sessions (%{customdata[1]:.1%})"
                                  "<br>%{customdata[2]} (%{customdata[3]:.1%})"
                                  "<br>%{customdata[4]:,} raw records<extra></extra>")
    # ~26 px per cloud row; bars capped at ~22 px thick however few rows there are
    h = max(180, 26 * len(names) + 90)
    fig.update_layout(height=h, bargap=max(0.35, 1 - 22 / ((h - 90) / max(len(names), 1))),
                      paper_bgcolor=_SURF, plot_bgcolor=_SURF, margin=dict(l=20, r=70, t=60, b=30),
                      font=dict(family="system-ui, sans-serif", size=12, color=_INK),
                      hoverlabel=dict(bgcolor=_SURF, font_color=_INK),
                      title=dict(text=title, x=0.01, xref="container", xanchor="left", font=dict(size=15, color=_INK)))
    # SI ticks; bytes get a "B" suffix so 1G reads 1 GB, not plotly's "1B" (= billion)
    fig.update_xaxes(type="log" if log else "linear", tickformat="~s", dtick=1 if log else None,
                     gridcolor=_GRID, gridwidth=1, zeroline=False, tickfont_color=_INK2, rangemode="tozero")
    fig.update_xaxes(row=1, col=2, ticksuffix="B")
    fig.update_yaxes(categoryorder="array", categoryarray=names[::-1], tickfont_color=_INK, ticksuffix="  ",
                     showgrid=False)
    fig.update_annotations(font=dict(size=13, color=_INK))
    return fig


_cldActW   = widgets.ToggleButtons(options=["all", "ACCEPT", "REJECT"], value="ACCEPT", description="action", style=_wsty)
_cldVals   = (vflowsAgg.loc[vflowsAgg["tag-direction"].eq("north-south").fillna(False), "tag-cloud"]
              .astype("string").fillna("(no cloud)").unique())
_cldVals   = sorted(_cldVals, key=str.lower)                                     # A-Z
_cldBoxes  = {v: widgets.Checkbox(value=True, description=v, indent=False,
                                  layout=widgets.Layout(width="auto", margin="0 14px 0 0"))
              for v in _cldVals}
_cldAllB   = widgets.Button(description="all",  layout=widgets.Layout(width="52px"))
_cldNoneB  = widgets.Button(description="none", layout=widgets.Layout(width="52px"))
_cldScaleW = widgets.ToggleButtons(options=["linear", "log"], description="x scale", style=_wsty)
_cldTopW   = widgets.Dropdown(options=[("top 10", 10), ("top 20", 20), ("all", 0)], value=CLD_TOP_N,
                              description="show", style=_wsty, layout=widgets.Layout(width="180px"))
_cldOut    = widgets.Output()          # cleared and redrawn on each change, so nothing piles up
_cldTbl    = widgets.HTML()
cloudSummary = None


def _cldUpdate(_=None):
    global cloudSummary
    sel = vflowsAgg[vflowsAgg["tag-direction"].eq("north-south").fillna(False)]
    picked = [v for v, b in _cldBoxes.items() if b.value]
    sel = sel[sel["tag-cloud"].astype("string").fillna("(no cloud)").isin(picked).to_numpy()]
    if _cldActW.value != "all":
        sel = sel[sel["action"].eq(_cldActW.value).fillna(False)]

    s = (sel.assign(cloud=sel["tag-cloud"].astype("string").fillna("(no cloud)"))
         .groupby("cloud")
         .agg(sessions=("records", "size"), bytes=("bytes", "sum"), records=("records", "sum"))
         .sort_values(["sessions", "bytes"], ascending=False))
    s["bytes"] = s["bytes"].fillna(0).astype("int64")
    s["sessionShare"] = s["sessions"] / max(int(s["sessions"].sum()), 1)
    s["byteShare"]    = s["bytes"] / max(int(s["bytes"].sum()), 1)
    cloudSummary = s

    top    = s.head(_cldTopW.value) if _cldTopW.value else s
    names  = top.index.tolist()
    colors = [_CLD_COLORS.get(n, _CLD_OTHER) for n in names]
    cdata  = list(zip(top["sessions"], top["sessionShare"], top["bytes"].map(_fmtBytes), top["byteShare"], top["records"]))
    shown  = f"top {len(top)} of {len(s)} clouds" if len(top) < len(s) else f"all {len(s)} clouds"
    title  = (f"Sessions and bytes by tag-cloud &nbsp;<span style='font-size:12px;color:{_INK2}'>"
              f"{shown}  ·  {len(sel):,} north-south sessions, {_fmtBytes(int(sel['bytes'].sum()))}</span>")
    fig    = _cldFigure(names, colors, top["sessions"].tolist(), top["bytes"].tolist(), cdata,
                        _cldScaleW.value == "log", title)
    with _cldOut:
        _cldOut.clear_output(wait=True)
        display(fig)

    tbl = s.assign(bytes=s["bytes"].map(_fmtBytes), sessionShare=s["sessionShare"].map("{:.1%}".format),
                   byteShare=s["byteShare"].map("{:.1%}".format), sessions=s["sessions"].map("{:,}".format),
                   records=s["records"].map("{:,}".format))
    _cldTbl.value = (_TBL_CSS + "<details><summary style='font:13px system-ui,sans-serif;cursor:pointer'>table view</summary>"
                     + tbl.reset_index().to_html(index=False, border=0, escape=True).replace("<table", "<table class='flowTbl'")
                     + "</details>")


def _cldSafeUpdate(_=None):
    """Errors inside widget callbacks are swallowed by the front end -- show them instead."""
    try:
        _cldUpdate()
    except Exception as e:
        _cldTbl.value = (f"<div style='font:13px system-ui,sans-serif;color:#c62828'><b>!! chart update failed:</b> "
                         f"{html.escape(f'{type(e).__name__}: {e}')}</div>")


for _w in (_cldActW, _cldScaleW, _cldTopW, *_cldBoxes.values()):
    _w.observe(_cldSafeUpdate, names="value")


def _cldSetAll(on):
    """Tick / untick every box, then redraw once (not once per box)."""
    for b in _cldBoxes.values():
        b.unobserve(_cldSafeUpdate, names="value")
        b.value = on
        b.observe(_cldSafeUpdate, names="value")
    _cldSafeUpdate()


_cldAllB.on_click(lambda _: _cldSetAll(True))
_cldNoneB.on_click(lambda _: _cldSetAll(False))

_cldUpdate()
_cldBoxRow = widgets.HBox([widgets.Label("cloud", layout=widgets.Layout(width="70px")), _cldAllB, _cldNoneB,
                           widgets.HBox(list(_cldBoxes.values()), layout=widgets.Layout(flex_flow="row wrap"))])
display(widgets.VBox([_cldActW, _cldBoxRow, widgets.HBox([_cldScaleW, _cldTopW]), _cldOut, _cldTbl]))

In [56]:
# --- cell 21 ---
# Sessions and bytes by tag-service (vflowsAgg). Two panels on one shared service axis
# -- never one chart with two scales. Sorted by sessions. Colors follow the service, not
# its rank: the 4 busiest named services (over all of vflowsAgg, fixed when this cell
# runs) get their own color and keep it through every filter; other named services are
# light gray; red = remote access (SVC_RED -- SSH, RDP, VNC); yellow = "anomalous" (src
# and dst ports name different services); mid gray = port not in conf/. Every bar is
# also labeled, so color is never the only cue.
# Filters redraw the chart (a plain figure, not a FigureWidget -- see _svcFigure); the table under the chart holds the same numbers.
# Checkboxes pick which tag-cloud values to include (as in cell 22) -- one per cloud in
# the data plus "(no cloud)" (tag-cloud empty: unmatched public ends, east-west and
# address-less sessions), in alphabetical order, all ticked to start; "all" / "none" set every box.
# Opens on: action ACCEPT, traffic north-south, every cloud ticked, x scale linear, top 10.
# The chart shows the top N services by sessions (SVC_TOP_N, "show" dropdown); the table
# and `serviceSummary` always hold every service.

for _col, _where in (("tag-service", "cell 14a"), ("tag-cloud", "cells 12-14")):
    if _col not in vflowsAgg.columns:
        raise KeyError(f"vflowsAgg has no '{_col}' column -- re-run {_where} (or everything from cell 08) first")
from plotly.subplots import make_subplots

# validated (dataviz validator, all pairs incl. the red and yellow below): blue, magenta,
# violet, aqua. Orange and green are left out -- too close to the red; yellow is anomalous.
_SVC_PALETTE = ["#2a78d6", "#e87ba4", "#4a3aa7", "#1baf7a"]
_SVC_OTHER   = "#c3c2b7"                                                  # remaining named services
_SVC_RED_HEX = "#d03b3b"                                                  # status-critical
SVC_RED      = ["SSH", "RDP", "VNC"]                                      # tag-service names drawn red
_SVC_COLORS  = {"anomalous": "#eda100", "(unlisted port)": "#898781"}    # yellow, muted gray
_SVC_COLORS.update({n: _SVC_RED_HEX for n in SVC_RED})
_svcRank = (vflowsAgg["tag-service"].dropna().loc[lambda s: ~s.isin(list(_SVC_COLORS))]
            .value_counts().index[:len(_SVC_PALETTE)])
_SVC_COLORS.update(dict(zip(_svcRank, _SVC_PALETTE)))
_INK, _INK2, _GRID, _SURF = "#0b0b0b", "#52514e", "#e8e7e2", "#fcfcfb"
SVC_TOP_N = 10


def _fmtBytes(b):
    for u in ("B", "kB", "MB", "GB", "TB"):
        if abs(b) < 1000 or u == "TB":
            return f"{b:,.0f} {u}" if u == "B" else f"{b:,.1f} {u}"
        b /= 1000


def _svcFigure(names, colors, sessions, nbytes, cdata, log, title):
    """A plain (one-way) plotly figure. Drawn fresh on every change -- no FigureWidget,
    whose two-way sync with the VS Code front end can hang the notebook."""
    fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.10,
                        subplot_titles=("Sessions", "Bytes"))
    for col, vals, txt in ((1, sessions, [f"{v:,}" for v in sessions]),
                           (2, nbytes,   [_fmtBytes(v) for v in nbytes])):
        fig.add_bar(row=1, col=col, orientation="h", y=names, x=vals, text=txt, customdata=cdata,
                    marker=dict(color=colors, cornerradius=4), textposition="outside", cliponaxis=False,
                    textfont_color=_INK2, showlegend=False,
                    hovertemplate="<b>%{y}</b><br>%{customdata[0]:,} sessions (%{customdata[1]:.1%})"
                                  "<br>%{customdata[2]} (%{customdata[3]:.1%})"
                                  "<br>%{customdata[4]:,} raw records<extra></extra>")
    # ~26 px per service row; bars capped at ~22 px thick however few rows there are
    h = max(180, 26 * len(names) + 90)
    fig.update_layout(height=h, bargap=max(0.35, 1 - 22 / ((h - 90) / max(len(names), 1))),
                      paper_bgcolor=_SURF, plot_bgcolor=_SURF, margin=dict(l=20, r=70, t=60, b=30),
                      font=dict(family="system-ui, sans-serif", size=12, color=_INK),
                      hoverlabel=dict(bgcolor=_SURF, font_color=_INK),
                      title=dict(text=title, x=0.01, xref="container", xanchor="left", font=dict(size=15, color=_INK)))
    # SI ticks; bytes get a "B" suffix so 1G reads 1 GB, not plotly's "1B" (= billion)
    fig.update_xaxes(type="log" if log else "linear", tickformat="~s", dtick=1 if log else None,
                     gridcolor=_GRID, gridwidth=1, zeroline=False, tickfont_color=_INK2, rangemode="tozero")
    fig.update_xaxes(row=1, col=2, ticksuffix="B")
    fig.update_yaxes(categoryorder="array", categoryarray=names[::-1], tickfont_color=_INK, ticksuffix="  ",
                     showgrid=False)
    fig.update_annotations(font=dict(size=13, color=_INK))
    return fig


_svcActW   = widgets.ToggleButtons(options=["all", "ACCEPT", "REJECT"], value="ACCEPT", description="action", style=_wsty)
_svcTypW   = widgets.ToggleButtons(options=["all", "north-south", "east-west"], value="north-south",
                                   description="traffic", style=_wsty)
_svcScaleW = widgets.ToggleButtons(options=["linear", "log"], description="x scale", style=_wsty)
_svcCldVals  = sorted(vflowsAgg["tag-cloud"].astype("string").fillna("(no cloud)").unique(), key=str.lower)   # A-Z
_svcCldBoxes = {v: widgets.Checkbox(value=True, description=v, indent=False,
                                    layout=widgets.Layout(width="auto", margin="0 14px 0 0"))
                for v in _svcCldVals}
_svcCldAllB  = widgets.Button(description="all",  layout=widgets.Layout(width="52px"))
_svcCldNoneB = widgets.Button(description="none", layout=widgets.Layout(width="52px"))
_svcTopW   = widgets.Dropdown(options=[("top 10", 10), ("top 20", 20), ("all", 0)], value=SVC_TOP_N,
                              description="show", style=_wsty, layout=widgets.Layout(width="180px"))
_svcOut    = widgets.Output()          # cleared and redrawn on each change, so nothing piles up
_svcTbl    = widgets.HTML()
serviceSummary = None


def _svcUpdate(_=None):
    global serviceSummary
    sel = vflowsAgg
    picked = [v for v, b in _svcCldBoxes.items() if b.value]
    sel = sel[sel["tag-cloud"].astype("string").fillna("(no cloud)").isin(picked).to_numpy()]
    if _svcActW.value != "all":
        sel = sel[sel["action"].eq(_svcActW.value).fillna(False)]
    if _svcTypW.value != "all":
        sel = sel[sel["tag-direction"].eq(_svcTypW.value).fillna(False)]

    s = (sel.assign(service=sel["tag-service"].astype("string").fillna("(unlisted port)"))
         .groupby("service")
         .agg(sessions=("records", "size"), bytes=("bytes", "sum"), records=("records", "sum"))
         .sort_values(["sessions", "bytes"], ascending=False))
    s["bytes"] = s["bytes"].fillna(0).astype("int64")
    s["sessionShare"] = s["sessions"] / max(int(s["sessions"].sum()), 1)
    s["byteShare"]    = s["bytes"] / max(int(s["bytes"].sum()), 1)
    serviceSummary = s

    top    = s.head(_svcTopW.value) if _svcTopW.value else s
    names  = top.index.tolist()
    colors = [_SVC_COLORS.get(n, _SVC_OTHER) for n in names]
    cdata  = list(zip(top["sessions"], top["sessionShare"], top["bytes"].map(_fmtBytes), top["byteShare"], top["records"]))
    shown  = f"top {len(top)} of {len(s)} services" if len(top) < len(s) else f"all {len(s)} services"
    title  = (f"Sessions and bytes by tag-service &nbsp;<span style='font-size:12px;color:{_INK2}'>"
              f"{shown}  ·  {len(sel):,} sessions, {_fmtBytes(int(sel['bytes'].sum()))}</span>")
    fig    = _svcFigure(names, colors, top["sessions"].tolist(), top["bytes"].tolist(), cdata,
                        _svcScaleW.value == "log", title)
    with _svcOut:
        _svcOut.clear_output(wait=True)
        display(fig)

    tbl = s.assign(bytes=s["bytes"].map(_fmtBytes), sessionShare=s["sessionShare"].map("{:.1%}".format),
                   byteShare=s["byteShare"].map("{:.1%}".format), sessions=s["sessions"].map("{:,}".format),
                   records=s["records"].map("{:,}".format))
    _svcTbl.value = (_TBL_CSS + "<details><summary style='font:13px system-ui,sans-serif;cursor:pointer'>table view</summary>"
                     + tbl.reset_index().to_html(index=False, border=0, escape=True).replace("<table", "<table class='flowTbl'")
                     + "</details>")


def _svcSafeUpdate(_=None):
    """Errors inside widget callbacks are swallowed by the front end -- show them instead."""
    try:
        _svcUpdate()
    except Exception as e:
        _svcTbl.value = (f"<div style='font:13px system-ui,sans-serif;color:#c62828'><b>!! chart update failed:</b> "
                         f"{html.escape(f'{type(e).__name__}: {e}')}</div>")


for _w in (_svcActW, _svcTypW, _svcScaleW, _svcTopW, *_svcCldBoxes.values()):
    _w.observe(_svcSafeUpdate, names="value")


def _svcCldSetAll(on):
    """Tick / untick every box, then redraw once (not once per box)."""
    for b in _svcCldBoxes.values():
        b.unobserve(_svcSafeUpdate, names="value")
        b.value = on
        b.observe(_svcSafeUpdate, names="value")
    _svcSafeUpdate()


_svcCldAllB.on_click(lambda _: _svcCldSetAll(True))
_svcCldNoneB.on_click(lambda _: _svcCldSetAll(False))

_svcUpdate()
_svcCldRow = widgets.HBox([widgets.Label("cloud", layout=widgets.Layout(width="70px")), _svcCldAllB, _svcCldNoneB,
                           widgets.HBox(list(_svcCldBoxes.values()), layout=widgets.Layout(flex_flow="row wrap"))])
display(widgets.VBox([_svcActW, _svcTypW, _svcCldRow, widgets.HBox([_svcScaleW, _svcTopW]), _svcOut, _svcTbl]))

In [ ]:
# --- cell 17 ---
# Top sessions in vflowsAgg by bytes or by duration. Filter by tag-direction, flow-direction, action, and whether
# the session has a tag-org. The table is an HTML widget whose .value is reassigned
# on every change (no Output widget, so nothing accumulates).
# Opens on: traffic north-south, direction all, action ACCEPT, org all, 25 rows.
# Precomputed once, when this cell runs: every session falls in one group (tag-direction x
# flow-direction x action x has-tag-org, <NA> a value of its own), and each group keeps its
# session count, byte total, and top _TOP_MAX row positions by bytes and by duration. A
# selection is a set of groups, so a click only adds up their totals and ranks their few
# kept rows -- vflowsAgg isn't scanned again. Re-run this cell after reloading vflowsAgg.

_TOP_MAX = 250         # most rows the table shows -- the largest "rows" option

_CSS  = "font:13px system-ui,sans-serif;margin:6px 0"
_typW = widgets.ToggleButtons(options=["all", "north-south", "east-west"], value="north-south",
                              description="traffic", style=_wsty)
_dirW = widgets.ToggleButtons(options=["all", "egress", "ingress"], description="direction", style=_wsty,
                                 disabled="flow-direction" not in vflowsAgg.columns)   # v2/v3 logs: no field
_actW = widgets.ToggleButtons(options=["all", "ACCEPT", "REJECT"], value="ACCEPT", description="action", style=_wsty)
_lblW = widgets.ToggleButtons(options=["all", "org", "NA"], description="org", style=_wsty)
_srtW = widgets.ToggleButtons(options=["bytes", "duration"], description="sort by", style=_wsty)
_nW   = widgets.Dropdown(options=[25, 50, 100, 250], value=25, description="rows", style=_wsty,
                         layout=widgets.Layout(width="160px"))
_topTbl = widgets.HTML()

_topCols = TABLE_COLS          # column order: cell 16a

# --- precompute: one group per session, then per-group totals and top rows ---
_na = pd.Series(pd.NA, index=vflowsAgg.index, dtype="string")
_topKeys = pd.DataFrame({
    "typ": vflowsAgg["tag-direction"].astype("string"),
    "dir": vflowsAgg["flow-direction"].astype("string") if "flow-direction" in vflowsAgg.columns else _na,
    "act": vflowsAgg["action"].astype("string"),
    "org": vflowsAgg["tag-org"].notna(),
})
_topGid = _topKeys.groupby(list(_topKeys.columns), dropna=False, sort=False).ngroup().to_numpy()
_topGrp = (_topKeys.assign(gid=_topGid, bytes=vflowsAgg["bytes"])
           .groupby("gid").agg(typ=("typ", "first"), dir=("dir", "first"), act=("act", "first"),
                               org=("org", "first"), sessions=("org", "size"), bytes=("bytes", "sum")))


def _topRank(v):
    """v (float per row, NaN = unranked) -> each group's top _TOP_MAX rows as a frame of
    gid / pos / val, largest first, ties in table order."""
    pos = np.flatnonzero(~np.isnan(v))
    gid, val = _topGid[pos], v[pos]
    o = np.lexsort((pos, -val, gid))
    pos, gid, val = pos[o], gid[o], val[o]
    first = np.r_[0, np.flatnonzero(np.diff(gid)) + 1]              # each group's first row
    rank = np.arange(len(gid)) - np.repeat(first, np.diff(np.r_[first, len(gid)]))
    keep = rank < _TOP_MAX
    return pd.DataFrame({"gid": gid[keep], "pos": pos[keep], "val": val[keep]})

_num = lambda c: vflowsAgg[c].to_numpy("float64", na_value=np.nan)
_topRanked = {"bytes":    _topRank(_num("bytes")),
              "duration": _topRank(_num("end") - _num("start"))}


def _topUpdate(_=None):
    g = _topGrp
    if _typW.value != "all":
        g = g[g["typ"].eq(_typW.value).fillna(False)]
    if _dirW.value != "all":
        g = g[g["dir"].eq(_dirW.value).fillna(False)]
    if _actW.value != "all":
        g = g[g["act"].eq(_actW.value).fillna(False)]
    if _lblW.value != "all":
        g = g[g["org"].eq(_lblW.value == "org")]

    r = _topRanked[_srtW.value]
    r = r[r["gid"].isin(g.index)].sort_values(["val", "pos"], ascending=[False, True]).head(_nW.value)
    top = (vflowsAgg.take(r["pos"].to_numpy())
           .assign(mb=lambda d: (d["bytes"] / 1e6).round(2),
                   hours=lambda d: ((d["end"] - d["start"]) / 3600).round(2))
           [[c for c in _topCols if c in vflowsAgg.columns or c in ("mb", "hours")]])
    _txt = top.select_dtypes(exclude="number").columns            # to_html's na_rep skips None / <NA>
    top[_txt] = top[_txt].astype(object).mask(top[_txt].isna(), "-")
    _topTbl.value = (
        _TBL_CSS +
        f"<div style='{_CSS}'><b>{int(g['sessions'].sum()):,}</b> sessions, "
        f"<b>{g['bytes'].sum() / 1e6:,.1f} MB</b> match &mdash; top {len(top)} by {_srtW.value}</div>"
        + top.to_html(index=False, border=0, escape=True, na_rep="-").replace(
            "<table", "<table class='flowTbl'"))


for _w in (_typW, _dirW, _actW, _lblW, _srtW, _nW):
    _w.observe(_topUpdate, names="value")

_topUpdate()
display(widgets.VBox([_typW, _dirW, _actW, _lblW, widgets.HBox([_srtW, _nW]), _topTbl]))

In [ ]:
# --- cell 20 ---
# Query vflowsAgg on any combination of srcaddr, srcport, dstaddr, dstport, protocol,
# tag-direction, action, flow-direction, tag-org, tag-service and tag-label. Blank / "all"
# = no filter; filled-in fields AND together. Text fields take comma-separated terms that
# OR together, and a leading ! excludes a term:
#   addresses  10.0.0.5, 10.0.0.0/16, !10.0.0.9     (IPv4 / CIDR; IPv6 exact match)
#   ports      443, 8000-8999, !22
#   protocol   tcp, udp, icmp or numbers -- 6, 17, !1
#   org        substring, any case -- amazon, !known; NA = no tag-org
#   service    tag-service, same rules -- ssh, rdp, anomalous; NA = port not in conf/
#   label      tag-label, same rules -- beaconing; NA = untagged (set by beaconing.ipynb)
#   peer label    peer_label, same rules -- comcast, !amazon; NA = none
#   peer ASN      peer_asn: numbers or ranges, "AS" optional -- AS13335, 16509, 64512-65534; NA = none
#   peer country  2 letters = exact country code (US -- not Russia), longer = substring of
#                 the name (united, russia); NA = none
# The three peer fields need the Cymru-enriched sessions (flows-nbad-cymru cell 11) loaded
# as vflowsAgg; without its peer_* columns they're greyed out.
# "either addr" / "either port" match a term at src OR dst; a ! term there drops
# sessions with that value at either end (e.g. !10.0.0.9 = doesn't involve 10.0.0.9).
# Nothing runs until asked: Enter in any text box or the "run query" button runs it, and
# only once at least one filter is set -- every toggle starts at "all", so there is no
# preset query. Toggles only set filters. Sort by / rows re-rank the last result without re-filtering.
# Circuit breaker: the scan runs in chunks of _Q_CHUNK rows and stops at the _Q_CAP-th
# match. Past that, the result is the first _Q_CAP matches in table order -- ranked
# among themselves, not the top of everything -- and the header says so.
# The result's row positions are left in `queryIdx`; queryResult() returns those rows
# as a DataFrame, and `queryComplete` says whether they are every match.
# "clear filters" empties every field, sets the toggles to "all", and drops the result.
# Re-run this cell after reloading vflowsAgg: the address ints and tag codes are cached.

_Q_CAP   = 1000         # stop the scan at this many matches
_Q_CHUNK = 1_000_000    # rows tested per step of the scan

_qCSS = "font:13px system-ui,sans-serif;margin:6px 0"
_qTBL_CSS = ("<style>.flowTbl{font:12px ui-monospace,monospace;border-collapse:collapse}"
             ".flowTbl th,.flowTbl td{padding:3px 12px;text-align:right;white-space:nowrap}"
             ".flowTbl thead th{border-bottom:1px solid #888}"
             ".flowTbl tbody tr:nth-child(even){background:rgba(128,128,128,.08)}</style>")
_qSty  = {"description_width": "90px"}
_qCols = ["start_time", "end_time", "hours", "srcaddr", "srcport", "dstaddr", "dstport",
          "protocol", "tag-direction", "action", "flow-direction", "tag-org", "tag-service", "tag-label",
          "peer_label", "peer_asn", "peer_country", "mb", "packets", "records"]
_qProto = {"icmp": 1, "tcp": 6, "udp": 17, "gre": 47, "esp": 50, "icmpv6": 58}


# Per row: IPv4 address as int (the pre-processor's src-int / dst-int), or a sentinel
# above the IPv4 range where it's <NA> (IPv6 / no address).
_qInts = {"srcaddr": vflowsAgg["src-int"].to_numpy(dtype="uint64", na_value=1 << 40),
          "dstaddr": vflowsAgg["dst-int"].to_numpy(dtype="uint64", na_value=1 << 40)}
_qCodes = {}   # col -> (codes, distinct values), on first use; tag columns repeat few values
_qAsn   = (pd.to_numeric(vflowsAgg["peer_asn"], errors="coerce").to_numpy("float64")   # Cymru stores text
           if "peer_asn" in vflowsAgg.columns else None)


def _qTerms(txt):
    """'a, !b, c' -> (['a', 'c'], ['b'])"""
    inc, exc = [], []
    for t in txt.replace(" ", ",").split(","):
        if t.strip("!"):
            (exc if t.startswith("!") else inc).append(t.lstrip("!"))
    return inc, exc


def _qApply(m, txt, termMask):
    """AND the OR of the included terms into m, then drop every excluded term."""
    inc, exc = _qTerms(txt)
    if inc:
        m &= reduce(np.logical_or, (termMask(t) for t in inc))
    for t in exc:
        m &= ~termMask(t)
    return m


# The mask builders below test one chunk of rows -- `sl`, a slice of vflowsAgg positions.

def _qAddrMask(col, sl, name=None):
    def f(t):
        if ":" in t:                                               # IPv6: exact match
            return vflowsAgg[col].iloc[sl].eq(t).fillna(False).to_numpy()
        try:
            net = ipaddress.IPv4Network(t, strict=False)
        except ValueError:
            raise ValueError(f"{name or col}: {t!r} is not an IPv4 address or CIDR") from None
        v = _qInts[col][sl]
        return (v >= int(net.network_address)) & (v <= int(net.broadcast_address))
    return f


def _qNumMask(col, sl, names=None, name=None):
    def f(t):
        t = (names or {}).get(t.lower(), t)
        try:
            lo, _, hi = str(t).partition("-")
            lo, hi = int(lo), int(hi or lo)
        except ValueError:
            raise ValueError(f"{name or col}: {t!r} is not a number or range"
                             + (f" (or one of {', '.join(names)})" if names else "")) from None
        return vflowsAgg[col].iloc[sl].between(lo, hi).fillna(False).to_numpy()
    return f


def _qEither(maskA, maskB):
    """Term matches if it matches either end."""
    return lambda t: maskA(t) | maskB(t)


def _qTagMask(col, sl):
    """Substring match, any case; NA = empty. A column that doesn't exist matches nothing.
    Matches against the column's distinct values, then maps the hits back to rows."""
    def f(t):
        if col not in vflowsAgg.columns:
            return np.full(sl.stop - sl.start, t.upper() == "NA")
        if col not in _qCodes:
            _qCodes[col] = pd.factorize(vflowsAgg[col])
        codes, uniq = _qCodes[col]
        if t.upper() == "NA":
            return codes[sl] < 0
        hit = (pd.Series(uniq, dtype="string").str.contains(t, case=False, regex=False)
               .fillna(False).to_numpy(bool))
        return np.append(hit, False)[codes[sl]]                    # code -1 (NA) -> False
    return f


def _qAsnMask(sl):
    """peer_asn: a number or lo-hi range, "AS" optional; NA = no ASN."""
    def f(t):
        if _qAsn is None:
            return np.full(sl.stop - sl.start, t.upper() == "NA")
        v = _qAsn[sl]
        if t.upper() == "NA":
            return np.isnan(v)
        try:
            lo, _, hi = re.sub(r"(?i)as", "", t).partition("-")
            lo, hi = int(lo), int(hi or lo)
        except ValueError:
            raise ValueError(f"peer ASN: {t!r} is not an AS number or range") from None
        return (v >= lo) & (v <= hi)                               # NaN compares False
    return f


def _qCountryMask(sl):
    """2 letters = exact peer_cc; longer = substring of peer_country; NA = no country."""
    name = _qTagMask("peer_country", sl)
    def f(t):
        if len(t) != 2 or t.upper() == "NA" or "peer_cc" not in vflowsAgg.columns:
            return name(t)
        if "peer_cc" not in _qCodes:
            _qCodes["peer_cc"] = pd.factorize(vflowsAgg["peer_cc"])
        codes, uniq = _qCodes["peer_cc"]
        hit = pd.Series(uniq, dtype="string").str.upper().eq(t.upper()).fillna(False).to_numpy(bool)
        return np.append(hit, False)[codes[sl]]
    return f


def _qText(desc, ph, col=None):
    """A filter box; greyed out when it filters a column (col) vflowsAgg doesn't have."""
    off = col is not None and col not in vflowsAgg.columns
    return widgets.Text(description=desc, style=_qSty, disabled=off,
                        placeholder="needs the Cymru-enriched sessions" if off else ph,
                        layout=widgets.Layout(width="420px"))

_qSrcW   = _qText("srcaddr",  "10.0.0.5, 10.0.0.0/16, !10.0.0.9")
_qSportW = _qText("srcport",  "443, 8000-8999, !22")
_qDstW   = _qText("dstaddr",  "10.0.0.5, 10.0.0.0/16, !10.0.0.9")
_qDportW = _qText("dstport",  "443, 8000-8999, !22")
_qAnyAW  = _qText("either addr", "src OR dst: 10.0.0.5, 10.0.0.0/16, !10.0.0.9")
_qAnyPW  = _qText("either port", "src OR dst: 443, 8000-8999, !22")
_qProtW  = _qText("protocol", "tcp, udp, icmp, 6, !17")
_qLblW   = _qText("org",      "amazon, !known, NA")
_qSvcW   = _qText("service",  "ssh, rdp, anomalous, NA")
_qTagW   = _qText("label",    "beaconing, NA")
_qPLblW  = _qText("peer label",   "comcast, !amazon, NA", "peer_label")
_qPAsnW  = _qText("peer ASN",     "AS13335, 16509, !8075, NA", "peer_asn")
_qPCtyW  = _qText("peer country", "US, russia, !CN, NA", "peer_country")
_qTypW   = widgets.ToggleButtons(options=["all", "north-south", "east-west", "NA"],
                                 description="traffic", style=_qSty)
_qActW   = widgets.ToggleButtons(options=["all", "ACCEPT", "REJECT", "NA"],
                                 description="action", style=_qSty)
_qDirW   = widgets.ToggleButtons(options=["all", "egress", "ingress", "NA"], description="direction", style=_qSty,
                                 disabled="flow-direction" not in vflowsAgg.columns)   # v2/v3 logs: no field
_qSrtW   = widgets.Dropdown(options=["bytes", "duration", "packets", "records", "latest", "earliest"],
                            description="sort by", style=_qSty, layout=widgets.Layout(width="220px"))
_qNW     = widgets.Dropdown(options=[25, 50, 100, 250, 1000], value=25, description="rows", style=_qSty,
                            layout=widgets.Layout(width="160px"))
_qRunB   = widgets.Button(description="run query", button_style="primary", layout=widgets.Layout(width="120px"))
_qClrB   = widgets.Button(description="clear filters", layout=widgets.Layout(width="120px"))
_qTbl    = widgets.HTML()
_qTextWs = (_qSrcW, _qSportW, _qDstW, _qDportW, _qAnyAW, _qAnyPW, _qProtW, _qLblW, _qSvcW, _qTagW,
            _qPLblW, _qPAsnW, _qPCtyW)
_qIdle   = (f"<div style='{_qCSS};color:#52514e'>Set filters, then press Enter in a text box "
            "or click <b>run query</b>.</div>")
queryIdx, queryComplete, _qScanned = None, None, 0


def queryResult():
    """The last query's rows as a DataFrame -- every match, or the first _Q_CAP when
    queryComplete is False."""
    if queryIdx is None:
        raise ValueError("no query has run yet -- run one in cell 20 first")
    return vflowsAgg.take(queryIdx)


def _qMask(sl):
    """Bool per row of the chunk sl: does it match every filter?"""
    m = np.ones(sl.stop - sl.start, dtype=bool)
    m = _qApply(m, _qSrcW.value,   _qAddrMask("srcaddr", sl))
    m = _qApply(m, _qDstW.value,   _qAddrMask("dstaddr", sl))
    m = _qApply(m, _qSportW.value, _qNumMask("srcport", sl))
    m = _qApply(m, _qDportW.value, _qNumMask("dstport", sl))
    m = _qApply(m, _qAnyAW.value,  _qEither(_qAddrMask("srcaddr", sl, "either addr"),
                                           _qAddrMask("dstaddr", sl, "either addr")))
    m = _qApply(m, _qAnyPW.value,  _qEither(_qNumMask("srcport", sl, name="either port"),
                                           _qNumMask("dstport", sl, name="either port")))
    m = _qApply(m, _qProtW.value,  _qNumMask("protocol", sl, _qProto))
    m = _qApply(m, _qLblW.value,   _qTagMask("tag-org", sl))
    m = _qApply(m, _qSvcW.value,   _qTagMask("tag-service", sl))
    m = _qApply(m, _qTagW.value,   _qTagMask("tag-label", sl))
    m = _qApply(m, _qPLblW.value,  _qTagMask("peer_label", sl))
    m = _qApply(m, _qPAsnW.value,  _qAsnMask(sl))
    m = _qApply(m, _qPCtyW.value,  _qCountryMask(sl))
    for w, col in ((_qTypW, "tag-direction"), (_qActW, "action"), (_qDirW, "flow-direction")):
        if col not in vflowsAgg.columns or w.value == "all":
            continue
        m &= (_qTagMask(col, sl)("NA") if w.value == "NA"
              else vflowsAgg[col].iloc[sl].eq(w.value).fillna(False).to_numpy())
    return m


def _qScan():
    """Matching row positions in table order, stopping at the _Q_CAP-th match.
    -> (positions, complete?, rows scanned)"""
    found, n, start = [], 0, 0
    while start < len(vflowsAgg):
        sl = slice(start, min(start + _Q_CHUNK, len(vflowsAgg)))
        hits = np.flatnonzero(_qMask(sl)) + start
        found.append(hits)
        n += len(hits)
        start = sl.stop
        if n > _Q_CAP:                                             # one past the cap: more exist
            return np.concatenate(found)[:_Q_CAP], False, start
    return (np.concatenate(found) if found else np.array([], dtype=np.int64)), True, start


def _qTop(idx, n):
    """Positions in idx ranked by the sort setting, NA last, ties in table order; first n."""
    key, asc = {"duration": ("_dur", False), "latest": ("start", False),
                "earliest": ("start", True)}.get(_qSrtW.value, (_qSrtW.value, False))
    num = lambda c: vflowsAgg[c].take(idx).to_numpy("float64", na_value=np.nan)
    v = num("end") - num("start") if key == "_dur" else num(key)
    v = np.where(np.isnan(v), np.inf, v if asc else -v)           # smallest first either way
    return idx[np.lexsort((idx, v))[:n]]


def _qShow(_=None):
    if queryIdx is None:
        return
    top = (vflowsAgg.take(_qTop(queryIdx, _qNW.value))
           .assign(mb=lambda d: (d["bytes"] / 1e6).round(2),
                   hours=lambda d: ((d["end"] - d["start"]) / 3600).round(2))
           [[c for c in _qCols if c in vflowsAgg.columns or c in ("mb", "hours")]])
    _txt = top.select_dtypes(exclude="number").columns            # to_html's na_rep skips None / <NA>
    top[_txt] = top[_txt].astype(object).mask(top[_txt].isna(), "-")
    n, srt = len(queryIdx), _qSrtW.value
    tot = (f"{vflowsAgg['bytes'].take(queryIdx).sum() / 1e6:,.1f} MB, "
           f"{vflowsAgg['records'].take(queryIdx).sum():,} records")
    if not queryComplete:
        head = (f"<div style='{_qCSS};color:#c62828'><b>More than {_Q_CAP:,} sessions match</b> &mdash; "
                f"the query stopped at the first {_Q_CAP:,} found, after scanning "
                f"{_qScanned / len(vflowsAgg):.0%} of sessions. These are in table order, not "
                f"the top {_Q_CAP:,} by {srt}: showing {len(top):,} of them by {srt} "
                f"({tot}). Narrow the filters for a complete result.</div>")
    elif n == 0:
        _qTbl.value = f"<div style='{_qCSS}'><b>No sessions match.</b></div>"
        return
    elif len(top) == n:
        head = (f"<div style='{_qCSS}'><b>{n:,}</b> sessions match ({tot}) &mdash; "
                f"all shown, by {srt}</div>")
    else:
        head = (f"<div style='{_qCSS}'><b>{n:,}</b> sessions match ({tot}) &mdash; "
                f"showing the top {len(top):,} by {srt}; raise rows to see all {n:,}</div>")
    _qTbl.value = (_qTBL_CSS + head
                   + top.to_html(index=False, border=0, escape=True, na_rep="-").replace(
                       "<table", "<table class='flowTbl'"))


def _qRun(_=None):
    global queryIdx, queryComplete, _qScanned
    if (not any(w.value.strip(" ,!") for w in _qTextWs)
            and all(w.value == "all" for w in (_qTypW, _qActW, _qDirW))):
        _qTbl.value = (f"<div style='{_qCSS};color:#52514e'>No filters set &mdash; fill in a field "
                       "or pick a toggle, then run the query.</div>")
        return
    _qRunB.disabled = True
    _qTbl.value = f"<div style='{_qCSS};color:#52514e'>running query&hellip;</div>"
    try:
        queryIdx, queryComplete, _qScanned = _qScan()
    except ValueError as e:
        queryIdx = queryComplete = None
        _qTbl.value = f"<div style='{_qCSS};color:#c62828'><b>!!</b> {html.escape(str(e))}</div>"
        return
    finally:
        _qRunB.disabled = False
    _qShow()


def _qClear(_=None):
    global queryIdx, queryComplete
    for w in _qTextWs + (_qTypW, _qActW, _qDirW):
        w.value = "all" if isinstance(w, widgets.ToggleButtons) else ""
    queryIdx = queryComplete = None
    _qTbl.value = _qIdle


# Enter in a text box runs the query. on_submit is deprecated in ipywidgets 8 but is the
# only Enter-only hook: observing value with continuous_update=False also fires when the
# box loses focus, which would run a query every time you tab between fields.
with warnings.catch_warnings():
    warnings.simplefilter("ignore", DeprecationWarning)
    for _w in _qTextWs:
        _w.on_submit(_qRun)
for _w in (_qSrtW, _qNW):
    _w.observe(_qShow, names="value")
_qRunB.on_click(_qRun)
_qClrB.on_click(_qClear)

_qTbl.value = _qIdle
display(widgets.VBox([
    widgets.HBox([_qSrcW, _qSportW]),
    widgets.HBox([_qDstW, _qDportW]),
    widgets.HBox([_qAnyAW, _qAnyPW]),
    widgets.HBox([_qProtW, _qLblW]),
    widgets.HBox([_qSvcW, _qTagW]),
    widgets.HBox([_qPLblW, _qPAsnW]),
    widgets.HBox([_qPCtyW]),
    _qTypW, _qActW, _qDirW,
    widgets.HBox([_qRunB, _qSrtW, _qNW, _qClrB]),
    _qTbl]))